# Adaptive Learning Platform - Phase 1

This notebook prototypes accessible text transformation and quiz generation before API wiring. Run the cells from top to bottom.

## 1. SETUP

Imports, environment configuration, the shared LLM wrapper, and sample input.

In [1]:
import json
from dotenv import load_dotenv
load_dotenv()
import re
from typing import Any, Dict, List
import os
import pdfplumber
import requests

try:
    from groq import Groq
except ImportError:
    Groq = None

GROQ_API_KEY = os.getenv("GROQ_API_KEY") or os.getenv("GROQ_KEY")
GROQ_MODEL = os.getenv("GROQ_MODEL")
GROQ_CLIENT = None

if GROQ_API_KEY and Groq is not None:
    GROQ_CLIENT = Groq(api_key=GROQ_API_KEY)
    if not GROQ_MODEL:
        available_models = {model.id for model in GROQ_CLIENT.models.list().data}
        preferred_models = (
            "llama-3.3-70b-versatile",
            "llama-3.1-8b-instant",
            "openai/gpt-oss-20b",
            "openai/gpt-oss-120b",
        )
        GROQ_MODEL = next(
            (model for model in preferred_models if model in available_models),
            next((model for model in available_models if "llama" in model or "gpt" in model), None),
        )
    if not GROQ_MODEL:
        raise RuntimeError("No text-generation model is available for this Groq project. Set GROQ_MODEL in .env.")


def call_llm(prompt: str) -> str:
    """Generate text with Groq, or use the local fallback when no key is configured."""
    if GROQ_API_KEY:
        if GROQ_CLIENT is None:
            raise ImportError("Install the groq package in this notebook before using GROQ_API_KEY.")
        request_options = {
            "model": GROQ_MODEL,
            "messages": [{"role": "user", "content": prompt}],
            "temperature": 0.2,
            "max_completion_tokens": 4096,
        }
        if "QUIZ_JSON" in prompt or "CHUNK_JSON" in prompt:
            request_options["response_format"] = {"type": "json_object"}
        if GROQ_MODEL.startswith("openai/"):
            request_options["reasoning_effort"] = "low"
        response = GROQ_CLIENT.chat.completions.create(**request_options)
        return response.choices[0].message.content or ""

    if "QUIZ_JSON" in prompt:
        return json.dumps({
            "question": "What do plants use photosynthesis to produce?",
            "options": ["Glucose", "Sound", "Salt", "Metal"],
            "answer": "Glucose",
            "explanation": "Photosynthesis produces glucose, which stores chemical energy.",
        })
    if "CHUNK_JSON" in prompt:
        sentences = re.split(r"(?<=[.!?])\s+", prompt.split("TEXT:", 1)[-1].strip())
        return json.dumps({"chunks": [" ".join(sentences[index:index + 3]) for index in range(0, len(sentences), 3)]})
    return (
        "Photosynthesis lets plants make food from sunlight. Chlorophyll captures light energy. "
        "Plants use water and carbon dioxide to produce glucose and release oxygen."
    )

print("Groq SDK available:", Groq is not None)
print("Groq API configured:", bool(GROQ_API_KEY))
print("Groq model:", GROQ_MODEL or "local fallback")

Groq SDK available: True
Groq API configured: True
Groq model: openai/gpt-oss-20b


## 2. PDF EXTRACTION

PDF text extraction uses `pdfplumber`, with a page-level fallback for image-only or otherwise empty pages.

In [2]:
def extract_text_and_tables_from_pdf(filepath: str) -> tuple[str, List[List[List[str]]]]:
    """Extract prose text and tables separately so table rows don't bleed into paragraphs.

    Table regions are detected first via find_tables(), then excluded from the
    text extraction pass using page.filter(), so extract_text() only sees prose.
    """
    pages_text: List[str] = []
    pages_tables: List[List[List[List[str]]]] = []

    try:
        with pdfplumber.open(filepath) as pdf:
            for page_number, page in enumerate(pdf.pages, start=1):
                found_tables = page.find_tables()
                table_bboxes = [t.bbox for t in found_tables]  # (x0, top, x1, bottom)

                def is_inside_a_table(obj, boxes=table_bboxes) -> bool:
                    for (tx0, ttop, tx1, tbottom) in boxes:
                        if obj["x0"] >= tx0 and obj["x1"] <= tx1 and obj["top"] >= ttop and obj["bottom"] <= tbottom:
                            return True
                    return False

                prose_only_page = page.filter(lambda obj: not is_inside_a_table(obj))
                page_text = (prose_only_page.extract_text() or "").strip()
                if not page_text:
                    page_text = f"[No extractable prose text found on page {page_number}; OCR may be required.]"
                pages_text.append(page_text)

                pages_tables.append([t.extract() for t in found_tables])
    except FileNotFoundError:
        raise FileNotFoundError(f"PDF file not found: {filepath}")
    except Exception as error:
        raise RuntimeError(f"Could not extract PDF content from {filepath}: {error}") from error

    return "\n\n".join(pages_text), pages_tables


PDF_PATH = "./sample_input.pdf"

# No fallback: if extraction fails, stop here rather than silently using other text.
SOURCE_TEXT, SOURCE_TABLES = extract_text_and_tables_from_pdf(PDF_PATH)

print("--- Extracted prose text ---")
print(SOURCE_TEXT)

print("\n--- Extracted tables ---")
for page_num, page_tables in enumerate(SOURCE_TABLES, start=1):
    for table_num, table in enumerate(page_tables, start=1):
        print(f"\nPage {page_num}, table {table_num}:")
        for row in table:
            print(row)

print("\nSource words:", len(SOURCE_TEXT.split()))
print("Tables found:", sum(len(t) for t in SOURCE_TABLES))

--- Extracted prose text ---
Chapter 7: Photosynthesis and Energy Flow
Grade 9 Biology · Unit 3: Plant Systems
7.1 What is Photosynthesis?
Photosynthesis is the biochemical process by which green plants, algae, and certain bacteria
convert light energy, typically from the sun, into chemical energy stored in glucose molecules.
This process is fundamental to almost all life on Earth, as it forms the base of most food chains
and is responsible for producing the oxygen that most organisms depend on for cellular
respiration. The overall chemical equation for photosynthesis can be summarized as carbon
dioxide plus water, in the presence of light energy, yielding glucose and oxygen.
The process takes place primarily in the chloroplasts of plant cells, specialized organelles that
contain a green pigment called chlorophyll. Chlorophyll is essential because it absorbs light most
efficiently in the blue and red wavelengths of the visible spectrum, while reflecting green light,
which is why most p

In [3]:
import pytesseract
from PIL import Image

# Windows only — uncomment and set your actual install path:
# pytesseract.pytesseract.tesseract_cmd = r"C:\Program Files\Tesseract-OCR\tesseract.exe"
filepath = r"D:\hackathon_projects\bit_build\sample_input_image.png"

def extract_text_from_image(filepath: str) -> str:
    """Extract text from an image (screenshot, photo of a worksheet, etc.) via OCR."""
    try:
        image = Image.open(filepath)
        text = pytesseract.image_to_string(image).strip()
        if not text:
            return "[No text detected in image; it may be blank or too low-quality for OCR.]"
        return text
    except FileNotFoundError:
        raise FileNotFoundError(f"Image file not found: {filepath}")
    except Exception as error:
        raise RuntimeError(f"Could not extract text from image {filepath}: {error}") from error

print("--- Extracted image text ---")
print(extract_text_from_image(filepath))

--- Extracted image text ---
Chapter 5: Newton's Laws of Motion

Grade 9 Physics - Unit 2: Forces and Motion

5.1 The First Law

Newton's First Law states that an object at rest stays at rest, and

an object in motion stays in motion at a constant velocity, unless
acted on by an unbalanced external force. This property of matter is
called inertia. A book resting on a table remains still because the
forces acting on it, gravity pulling down and the table pushing up,
are balanced and cancel each other out.

5.2 The Second Law

The Second Law describes what happens when an unbalanced force does
act on an object: the object accelerates in the direction of the net

force. This relationship is written as force equals mass multiplied

by acceleration. A heavier object requires more force to achieve the
same acceleration as a lighter one, which is why pushing a loaded

cart takes more effort than pushing an empty one.

5.3 Key Terms
* Inertia: the tendency of an object to resist a change in it

## 3. PROFILE-BASED TRANSFORMATION

Prompt constants are deliberately separate from the transformation function so they can be edited without changing control flow.

In [4]:
DYSLEXIA_PROMPT = """Rewrite the text below for a reader with dyslexia. Use short, simple sentences and clear wording. Preserve every fact, relationship, number, and cause-and-effect detail. Do not add facts. Return only the rewritten text.

TEXT:
{text}"""

COGNITIVE_LOAD_PROMPT = """Split the text below into an ordered JSON object with one key, chunks. The chunks value must be an array of digestible chunks. Each chunk must contain 2 to 4 complete sentences. Preserve all original content and facts, do not summarize, and do not add facts. Return only valid JSON. Include the marker CHUNK_JSON nowhere except in this instruction context.

TEXT:
{text}"""


def _parse_json_response(response: str) -> Any:
    """Remove optional Markdown fences and parse a JSON response."""
    cleaned = re.sub(r"^\s*```(?:json)?\s*|\s*```\s*$", "", response.strip(), flags=re.IGNORECASE)
    return json.loads(cleaned)

In [5]:
PREFERENCE_PARSE_PROMPT = """A user described their accessibility needs in their own words below.
Map their description to exactly one of these profiles: dyslexia, cognitive_load, low_vision.
If multiple seem to apply, pick the single best match. Return valid JSON only with these keys:
profile (one of the three values above), reason (one short sentence explaining the match).

USER DESCRIPTION:
{text}"""

def parse_user_preference(user_text: str) -> Dict[str, Any]:
    """Convert a free-text accessibility description into a structured profile."""
    response = call_llm(PREFERENCE_PARSE_PROMPT.format(text=user_text))
    result = _parse_json_response(response)
    if result.get("profile") not in ("dyslexia", "cognitive_load", "low_vision"):
        raise ValueError(f"Unrecognized profile returned: {result.get('profile')}")
    return result

In [6]:
def transform_text(text: str, profile: str) -> Dict[str, Any]:
    """Transform text according to an accessibility profile."""
    if profile == "low_vision":
        return {"profile": profile, "text": text, "formatting": {"font_size_multiplier": 1.5, "contrast_mode": "high"}}
    if profile == "dyslexia":
        rewritten = call_llm(DYSLEXIA_PROMPT.format(text=text))
        if not rewritten.strip():
            raise ValueError("Groq returned an empty dyslexia transformation.")
        return {"profile": profile, "text": rewritten.strip()}
    if profile == "cognitive_load":
        response = call_llm(COGNITIVE_LOAD_PROMPT.format(text=text))
        parsed = _parse_json_response(response)
        chunks = parsed.get("chunks") if isinstance(parsed, dict) else parsed
        if not isinstance(chunks, list) or not all(isinstance(chunk, str) and chunk.strip() for chunk in chunks):
            raise ValueError("Cognitive-load response must contain a JSON chunks array of strings.")
        return {"profile": profile, "chunks": chunks}
    raise ValueError("profile must be dyslexia, low_vision, or cognitive_load")

for profile in ("dyslexia", "low_vision", "cognitive_load"):
    print(f"\n--- {profile.upper()} ---")
    print(json.dumps(transform_text(SOURCE_TEXT, profile), indent=2, ensure_ascii=False))


--- DYSLEXIA ---
{
  "profile": "dyslexia",
  "text": "Chapter 7: Photosynthesis and Energy Flow  \nGrade 9 Biology · Unit 3: Plant Systems  \n\n7.1 What is Photosynthesis?  \nPhotosynthesis is a chemical process.  \nGreen plants, algae, and some bacteria do it.  \nThey use light energy, usually from the sun.  \nThey turn that light into chemical energy.  \nThe energy is stored in glucose molecules.  \nThis process is very important.  \nIt is the base of most food chains.  \nIt also makes the oxygen that most living things need.  \nThe overall equation is:  \ncarbon dioxide + water + light → glucose + oxygen.  \n\nThe process happens mainly in chloroplasts.  \nChloroplasts are special parts of plant cells.  \nThey contain a green pigment called chlorophyll.  \nChlorophyll absorbs light best in the blue and red parts of the spectrum.  \nIt reflects green light.  \nThat is why plants look green.  \n\nInside the chloroplast, photosynthesis has two stages.  \nEach stage happens in a diffe

In [7]:
import ipywidgets as widgets
from IPython.display import display

PROFILE_LABELS = {
    "I have dyslexia": "dyslexia",
    "I find long or dense text difficult": "cognitive_load",
    "I need larger, high-contrast text": "low_vision",
}

profile_selector = widgets.Dropdown(
    options=list(PROFILE_LABELS),
    value="I have dyslexia",
    description="I am dealing with:",
    layout=widgets.Layout(width="550px"),
)
text_input = widgets.Textarea(
    value=SOURCE_TEXT,   # was: value=SAMPLE_TEXT
    description="Text:",
    layout=widgets.Layout(width="700px", height="180px"),
)
generate_button = widgets.Button(description="Generate accessible version", button_style="primary")
transformation_output = widgets.Output()

def render_transformation(result: Dict[str, Any]) -> None:
    """Display the generated result in a form suited to the selected profile."""
    with transformation_output:
        transformation_output.clear_output()
        if result["profile"] == "cognitive_load":
            print("Generated digestible chunks:\n")
            for number, chunk in enumerate(result["chunks"], start=1):
                print(f"{number}. {chunk}\n")
        else:
            print(result["text"])
            if result["profile"] == "low_vision":
                print("\nDisplay settings: larger text (1.5x), high contrast")

def generate_selected_transformation(_button: widgets.Button) -> None:
    """Generate output from the user's selected profile and text."""
    text = text_input.value.strip()
    with transformation_output:
        transformation_output.clear_output()
        if not text:
            print("Please enter some text before generating an accessible version.")
            return
        try:
            selected_profile = PROFILE_LABELS[profile_selector.value]
            render_transformation(transform_text(text, selected_profile))
        except (ValueError, json.JSONDecodeError) as error:
            print(f"Could not generate the transformation: {error}")

generate_button.on_click(generate_selected_transformation)
preference_text_input = widgets.Textarea(
    placeholder="Or describe what you need in your own words, e.g. 'I have dyslexia and long paragraphs overwhelm me'",
    layout=widgets.Layout(width="700px", height="60px"),
)
parse_preference_button = widgets.Button(description="Detect profile from description")
preference_output = widgets.Output()

def detect_profile_from_text(_button):
    with preference_output:
        preference_output.clear_output()
        description = preference_text_input.value.strip()
        if not description:
            print("Enter a description first.")
            return
        try:
            result = parse_user_preference(description)
            print(f"Detected profile: {result['profile']}")
            print(f"Reason: {result['reason']}")
            # sync it into the existing dropdown so the rest of the flow just works
            matching_label = [label for label, value in PROFILE_LABELS.items() if value == result["profile"]][0]
            profile_selector.value = matching_label
        except (ValueError, json.JSONDecodeError) as error:
            print(f"Could not detect profile: {error}")

parse_preference_button.on_click(detect_profile_from_text)
display(widgets.VBox([preference_text_input, parse_preference_button, preference_output]))
display(widgets.VBox([profile_selector, text_input, generate_button, transformation_output]))

In [ ]:
from typing import Optional

In [8]:
# Integrated voice and visual learning features from adaptive_learning_2
import base64
import io
import textwrap
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from IPython.display import Javascript, HTML

VOICE_HELP_PROMPT = """You are an educational assistant. Answer using only the lesson content.
Do not invent facts. Keep the answer concise and clear.
If the answer is not in the lesson, say: The lesson does not provide enough information to answer that.

LESSON:
{lesson}

LEARNER QUESTION:
{question}"""

VISUAL_SPEC_PROMPT = """Analyse this lesson and create a visual explanation for the learner profile: {profile}.
Use only facts from the lesson. Return valid JSON with exactly these keys:
should_visualize, visual_type, title, description, why_helpful, nodes, edges, labels, data.
visual_type must be flowchart, timeline, process, graph, concept_map, or none.
nodes must be short labels. edges must contain integer [from, to] pairs. labels must match edges.
data is only for graph visuals and contains label/value objects.

LESSON:
{lesson}"""


def voice_ask(user_request: str, lesson_text: str = "") -> str:
    """Answer a learner question using only the current lesson."""
    lesson = (lesson_text or SOURCE_TEXT).strip()
    try:
        return call_llm(VOICE_HELP_PROMPT.format(lesson=lesson, question=user_request.strip())).strip()
    except Exception as error:
        return f"[Voice assistant error: {error}]"


def _fallback_visual_spec(lesson_text: str, profile: str) -> Dict[str, Any]:
    """Provide a local visual when no visual API response is available."""
    sentences = [part.strip() for part in re.split(r"(?<=[.!?])\s+", lesson_text) if part.strip()]
    nodes = [" ".join(textwrap.wrap(sentence, 24)) for sentence in sentences[:5]]
    return {
        "should_visualize": bool(nodes),
        "visual_type": "flowchart" if nodes else "none",
        "title": "Lesson sequence",
        "description": "Key lesson ideas in order.",
        "why_helpful": "A sequence makes the relationships between the main ideas easier to follow.",
        "nodes": nodes,
        "edges": [[index, index + 1] for index in range(max(0, len(nodes) - 1))],
        "labels": [""] * max(0, len(nodes) - 1),
        "data": [],
    }


def generate_visual_spec(lesson_text: str, profile: str) -> Dict[str, Any]:
    """Generate and validate a visual specification for the current lesson."""
    try:
        if GROQ_API_KEY:
            response = call_llm(VISUAL_SPEC_PROMPT.format(profile=profile, lesson=lesson_text[:6000]))
            spec = _parse_json_response(response)
        else:
            spec = _fallback_visual_spec(lesson_text, profile)
    except Exception:
        spec = _fallback_visual_spec(lesson_text, profile)

    required = {"should_visualize", "visual_type", "title", "description", "why_helpful", "nodes", "edges", "labels", "data"}
    valid_types = {"flowchart", "timeline", "process", "graph", "concept_map", "none"}
    if not isinstance(spec, dict) or not required.issubset(spec) or spec["visual_type"] not in valid_types:
        return {"error": "The visual response was not valid JSON."}
    node_count = len(spec.get("nodes", []))
    if any(not isinstance(edge, list) or len(edge) != 2 or not all(isinstance(index, int) and 0 <= index < node_count for index in edge) for edge in spec.get("edges", [])):
        return {"error": "The visual response contained invalid node links."}
    return spec


def _fig_to_base64(fig) -> str:
    buffer = io.BytesIO()
    fig.savefig(buffer, format="png", bbox_inches="tight", dpi=130)
    plt.close(fig)
    return base64.b64encode(buffer.getvalue()).decode("ascii")


def render_visual_spec(spec: Dict[str, Any], profile: str) -> Optional[str]:
    """Render supported visual specifications as a base64 PNG."""
    if "error" in spec or not spec.get("should_visualize"):
        return None
    nodes = spec.get("nodes", [])
    if spec.get("visual_type") == "graph" and spec.get("data"):
        labels = [item.get("label", "") for item in spec["data"]]
        values = [float(item.get("value", 0)) for item in spec["data"]]
        fig, axis = plt.subplots(figsize=(10, 5))
        axis.bar(labels, values, color="#93c5fd", edgecolor="#1d4ed8", linewidth=2)
        axis.set_title(spec.get("title", ""), fontsize=16, fontweight="bold")
        axis.tick_params(labelsize=12)
        return _fig_to_base64(fig)

    if not nodes:
        return None
    fig, axis = plt.subplots(figsize=(10, max(4, len(nodes) * 1.1)))
    axis.set_xlim(0, 1)
    axis.set_ylim(0, 1)
    axis.axis("off")
    box_width, box_height = 0.64, min(0.12, 0.78 / len(nodes))
    gap = (1 - box_height * len(nodes)) / (len(nodes) + 1)
    centers = []
    for index, node in enumerate(nodes):
        center = (0.5, 1 - gap - index * (box_height + gap) - box_height / 2)
        centers.append(center)
        box = mpatches.FancyBboxPatch(
            (center[0] - box_width / 2, center[1] - box_height / 2), box_width, box_height,
            boxstyle="round,pad=0.02", facecolor="#dcfce7", edgecolor="#166534", linewidth=2,
        )
        axis.add_patch(box)
        axis.text(center[0], center[1], node, ha="center", va="center", fontsize=12, fontweight="bold")
    for first, second in spec.get("edges", []):
        x0, y0 = centers[first]
        x1, y1 = centers[second]
        axis.annotate("", xy=(x1, y1 + box_height / 2), xytext=(x0, y0 - box_height / 2),
                      arrowprops={"arrowstyle": "->", "color": "#166534", "lw": 2})
    axis.set_title(spec.get("title", ""), fontsize=16, fontweight="bold", pad=10)
    return _fig_to_base64(fig)


integrated_voice_input = widgets.Text(placeholder="Type a question about the lesson", layout=widgets.Layout(width="600px"))
integrated_ask_button = widgets.Button(description="Ask", button_style="info")
integrated_speak_button = widgets.Button(description="Speak", button_style="primary")
integrated_read_button = widgets.Button(description="Read Aloud", button_style="success", layout=widgets.Layout(display="none"))
integrated_voice_output = widgets.Output()
integrated_visual_button = widgets.Button(description="Generate Visual", button_style="primary")
integrated_visual_output = widgets.Output()
integrated_last_response = [""]


def _run_integrated_voice(_button=None):
    question = integrated_voice_input.value.strip()
    with integrated_voice_output:
        integrated_voice_output.clear_output()
        if not question:
            print("Please enter a question first.")
            return
        response = voice_ask(question, text_input.value.strip() or SOURCE_TEXT)
        integrated_last_response[0] = response
        print(f"PRISM: {response}")
        integrated_read_button.layout.display = ""


def _read_integrated_response(_button=None):
    response = integrated_last_response[0]
    if response:
        display(Javascript(f"window.speechSynthesis.cancel(); window.speechSynthesis.speak(new SpeechSynthesisUtterance({json.dumps(response)}));"))


def _start_integrated_speech(_button=None):
    display(Javascript("""(function(){
        const Recognition = window.SpeechRecognition || window.webkitSpeechRecognition;
        if (!Recognition) { alert('Speech recognition is not supported in this browser.'); return; }
        const recognition = new Recognition();
        recognition.lang = 'en-US';
        recognition.onresult = event => {
            const text = event.results[0][0].transcript;
            const input = document.querySelector('input[placeholder="Type a question about the lesson"]');
            if (input) { input.value = text; input.dispatchEvent(new Event('input', {bubbles:true})); }
        };
        recognition.start();
    })();"""))


def _render_integrated_visual(_button=None):
    lesson = text_input.value.strip() or SOURCE_TEXT
    profile = PROFILE_LABELS[profile_selector.value]
    spec = generate_visual_spec(lesson, profile)
    with integrated_visual_output:
        integrated_visual_output.clear_output(wait=True)
        if "error" in spec:
            print(spec["error"])
            return
        image = render_visual_spec(spec, profile)
        display(HTML(f"<h4>{spec.get('title', 'Visual explanation')}</h4><p>{spec.get('description', '')}</p>"))
        if image:
            display(HTML(f"<img src='data:image/png;base64,{image}' style='max-width:100%;' alt='{spec.get('title', 'Visual explanation')}' />"))
        if spec.get("why_helpful"):
            print(f"Why this helps: {spec['why_helpful']}")


integrated_ask_button.on_click(_run_integrated_voice)
integrated_speak_button.on_click(_start_integrated_speech)
integrated_read_button.on_click(_read_integrated_response)
integrated_visual_button.on_click(_render_integrated_visual)

integrated_ui = widgets.VBox([
    widgets.HTML("<hr><h3>Voice and Visual Learning</h3>"),
    widgets.HBox([integrated_voice_input, integrated_ask_button, integrated_speak_button, integrated_read_button]),
    integrated_voice_output,
    integrated_visual_button,
    integrated_visual_output,
])
display(integrated_ui)

NameError: name 'Optional' is not defined

## 4. QUIZ GENERATION

Each chunk gets one LLM call. The parser accepts plain JSON and JSON wrapped in Markdown code fences.

In [ ]:
QUIZ_PROMPT = """Create one practice question based only on the chunk below. Adjust phrasing complexity to the learner profile: {profile}. Return valid JSON only with exactly these keys: question, options, answer, explanation. The options value must be an array of exactly 4 strings. The answer must exactly match one option. Do not use information outside the chunk. Include the marker QUIZ_JSON nowhere except in this instruction context.

CHUNK:
{chunk}"""

def generate_quiz(chunk: str, profile: str) -> Dict[str, Any]:
    """Generate and validate one profile-aware quiz question for a text chunk."""
    response = call_llm(QUIZ_PROMPT.format(chunk=chunk, profile=profile))
    quiz = _parse_json_response(response)
    required_keys = {"question", "options", "answer", "explanation"}
    if set(quiz) != required_keys or not isinstance(quiz["options"], list) or len(quiz["options"]) != 4:
        raise ValueError("Quiz response must contain question, four options, answer, and explanation.")
    if quiz["answer"] not in quiz["options"]:
        raise ValueError("Quiz answer must match one of the options.")
    return quiz

cognitive_chunks = transform_text(SOURCE_TEXT, "cognitive_load")["chunks"]
for index, chunk in enumerate(cognitive_chunks[:3], start=1):
    print(f"\n--- QUIZ FOR CHUNK {index} ---")
    print(json.dumps(generate_quiz(chunk, "cognitive_load"), indent=2))


--- QUIZ FOR CHUNK 1 ---
{
  "question": "Which of the following best describes the overall chemical equation for photosynthesis?",
  "options": [
    "Carbon dioxide + water + light energy \u2192 glucose + oxygen",
    "Glucose + oxygen \u2192 carbon dioxide + water + light energy",
    "Carbon dioxide + oxygen \u2192 glucose + water",
    "Water + light energy \u2192 carbon dioxide + glucose"
  ],
  "answer": "Carbon dioxide + water + light energy \u2192 glucose + oxygen",
  "explanation": "The process of photosynthesis uses light energy to convert carbon dioxide and water into glucose and oxygen, as stated in the provided text."
}

--- QUIZ FOR CHUNK 2 ---
{
  "question": "Which pigment in chloroplasts is responsible for absorbing light most efficiently in the blue and red wavelengths?",
  "options": [
    "Carotene",
    "Chlorophyll",
    "Xanthophyll",
    "Anthocyanin"
  ],
  "answer": "Chlorophyll",
  "explanation": "Chlorophyll is the green pigment in chloroplasts that absorb

## 5. END-TO-END TEST

This final cell runs raw text through all three profile paths and generates practice questions for the transformed content.

In [ ]:
def chunks_for_quiz(transformed: Dict[str, Any]) -> List[str]:
    """Normalize a transformed profile result into quiz-ready chunks."""
    if transformed["profile"] == "cognitive_load":
        return transformed["chunks"]
    return [transformed["text"]]

# Section 5 end-to-end test
pipeline_summary: Dict[str, Any] = {}
for profile in ("dyslexia", "low_vision", "cognitive_load"):
    transformed = transform_text(SOURCE_TEXT, profile)
    quiz_results = [generate_quiz(chunk, profile) for chunk in chunks_for_quiz(transformed)[:3]]
    pipeline_summary[profile] = {
        "transformed_keys": list(transformed.keys()),
        "quiz_count": len(quiz_results),
        "first_question": quiz_results[0]["question"] if quiz_results else None,
    }

print("\n=== FINAL PIPELINE SUMMARY ===")
print(json.dumps(pipeline_summary, indent=2))


=== FINAL PIPELINE SUMMARY ===
{
  "dyslexia": {
    "transformed_keys": [
      "profile",
      "text"
    ],
    "quiz_count": 1,
    "first_question": "Which part of the chloroplast is where the light\u2011dependent reactions of photosynthesis happen?"
  },
  "low_vision": {
    "transformed_keys": [
      "profile",
      "text",
      "formatting"
    ],
    "quiz_count": 1,
    "first_question": "Which of the following is produced during the light-dependent reactions of photosynthesis?"
  },
  "cognitive_load": {
    "transformed_keys": [
      "profile",
      "chunks"
    ],
    "quiz_count": 3,
    "first_question": "What are the main products of photosynthesis as described in the chunk?"
  }
}
